# Análise Exploratória - MICRODADOS INEP 2024
## Fluxo alinhado ao app.py

**Objetivo**: explorar a mesma base usada no Streamlit, sem recriar a base antiga do INEP.

**Fontes usadas**:
- `base.xlsx` para a base V-Educa
- `MICRODADOS_CADASTRO_CURSOS_2024_SLIM.csv` para o INEP 2024
- `MICRODADOS_ED_SUP_IES_2024.CSV` apenas como apoio, se necessário

**Estrutura**:
- Carregamento padronizado com os mesmos helpers do app
- Análises descritivas por região, modalidade e categoria administrativa
- Destaque V-Educa dentro do recorte INEP
- Exportação de resultados em CSV e resumo textual

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import app as dashboard_app

ROOT_DIR = dashboard_app.ROOT_DIR
MAIN_CSV_PATH = dashboard_app.INEP_MAIN_PATH
IES_CSV_PATH = ROOT_DIR / 'MICRODADOS_ED_SUP_IES_2024.CSV'
SLIM_CSV_PATH = dashboard_app.INEP_SLIM_PATH
OUTPUT_DIR = ROOT_DIR / 'outputs_inep'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Arquivo principal: {MAIN_CSV_PATH}')
print(f'Existe principal: {MAIN_CSV_PATH.exists()}')
print(f'Arquivo reduzido: {SLIM_CSV_PATH}')
print(f'Existe reduzido: {SLIM_CSV_PATH.exists()}')
print(f'Arquivo IES: {IES_CSV_PATH}')
print(f'Existe IES: {IES_CSV_PATH.exists()}')

alunos, meta = dashboard_app.load_data()
df_inep = dashboard_app.load_inep_data()

COLUNAS_ANALISE_DICIONARIO = [c for c in dashboard_app.INEP_NUMERIC_COLS if c in df_inep.columns]

print('')
print('✓ Base V-Educa carregada com os mesmos helpers do app.py')
print(f'Shape V-Educa: {alunos.shape}')
print(f'Shape INEP: {df_inep.shape}')
print(f'Memoria INEP: {df_inep.memory_usage(deep=True).sum() / (1024**2):.2f} MB')
print(f'Colunas numericas INEP disponiveis: {COLUNAS_ANALISE_DICIONARIO}')

Arquivo principal: c:\Users\alian\source\repos\VISAGIO\MICRODADOS_CADASTRO_CURSOS_2024.CSV
Existe principal: True
Tamanho principal: 431.87 MB
Arquivo IES: c:\Users\alian\source\repos\VISAGIO\MICRODADOS_ED_SUP_IES_2024.CSV
Existe IES: True
Arquivo reduzido: c:\Users\alian\source\repos\VISAGIO\MICRODADOS_CADASTRO_CURSOS_2024_SLIM.csv
Existe reduzido: True


## 1. Carregamento Otimizado da Base

Estratégia:
- Ler apenas colunas essenciais para reduzir uso de memória
- Definir tipos de dados corretos na leitura
- Usar sep=';' (ponto-e-vírgula)

In [2]:
# Primeira pass: ler apenas para inspecionar nomes das colunas do arquivo principal
print('Lendo cabecalho do arquivo principal...')
df_header = pd.read_csv(MAIN_CSV_PATH, sep=';', nrows=0, encoding='latin-1')
print(f'Total de colunas no principal: {len(df_header.columns)}')
print('\nPrimeiras 10 colunas:')
for i, col in enumerate(df_header.columns[:10]):
    print(f'  {i}: {col}')

Lendo cabecalho do arquivo principal...
Total de colunas no principal: 223

Primeiras 10 colunas:
  0: NU_ANO_CENSO
  1: NO_REGIAO
  2: CO_REGIAO
  3: NO_UF
  4: SG_UF
  5: CO_UF
  6: NO_MUNICIPIO
  7: CO_MUNICIPIO
  8: IN_CAPITAL
  9: TP_DIMENSAO


In [ ]:
print('DISTRIBUIÇÃO POR MODALIDADE')
print('=' * 80)
if 'TP_MODALIDADE_ENSINO' in df_inep.columns:
    print(df_inep['TP_MODALIDADE_ENSINO'].value_counts())
else:
    print('Coluna TP_MODALIDADE_ENSINO não disponível na base slim.')

print('')
print('DISTRIBUIÇÃO POR REGIÃO')
print('=' * 80)
if 'NO_REGIAO' in df_inep.columns:
    print(df_inep['NO_REGIAO'].value_counts())
else:
    print('Coluna NO_REGIAO não disponível na base slim.')

print('')
print('DISTRIBUIÇÃO POR CATEGORIA ADMINISTRATIVA')
print('=' * 80)
if 'TP_CATEGORIA_ADMINISTRATIVA' in df_inep.columns:
    print(df_inep['TP_CATEGORIA_ADMINISTRATIVA'].value_counts())
else:
    print('Coluna TP_CATEGORIA_ADMINISTRATIVA não disponível na base slim.')

Colunas alvo: 22
Colunas disponiveis no principal: 20

Carregando base reduzida...

✓ Base reduzida carregada com sucesso!
Shape: (722721, 23)
Memoria utilizada: 508.45 MB
Colunas de analise (dicionario) disponiveis: ['NU_ANO_CENSO', 'SG_UF', 'NO_REGIAO', 'CO_IES', 'TP_REDE', 'NO_CINE_ROTULO', 'NO_CINE_AREA_GERAL', 'TP_MODALIDADE_ENSINO', 'QT_CURSO', 'QT_VG_TOTAL', 'QT_VG_TOTAL_EAD', 'QT_ING', 'QT_ING_FEM', 'QT_ING_MASC', 'QT_MAT', 'QT_MAT_FEM', 'QT_MAT_MASC']
Colunas agregadas da base IES disponiveis: ['NO_IES', 'SG_IES']


## 1.1 Teste de Destaque V-Educa no INEP

Valida a regra usada no app para separar visualmente os grupos da V-Educa:

- `NO_IES` presente na base V-Educa
- `TP_MODALIDADE_ENSINO = 1` -> `VEDUCA Presencial`
- `TP_MODALIDADE_ENSINO = 2` -> `VEDUCA EAD`

O objetivo e testar a classificacao sem cortar registros do INEP.

In [ ]:
# Validação do destaque V-Educa no INEP seguindo a mesma regra do app.py
if 'TP_MODALIDADE_ENSINO' in df_inep.columns:
    df_teste = df_inep.copy()
    df_teste['__modal_label__'] = df_teste['TP_MODALIDADE_ENSINO'].map({1: 'Presencial', 2: 'Curso a distancia'})
    df_teste['__modal_label__'] = df_teste['__modal_label__'].fillna('Outros')

    in_veduca = pd.Series(False, index=df_teste.index)
    criterio_txt = ''

    if 'NO_MANTENEDORA' in df_teste.columns:
        mantenedora_txt = df_teste['NO_MANTENEDORA'].astype('string').fillna('').str.upper()
        in_veduca = mantenedora_txt.str.contains(r'V-EDUCA|VEDUCA', regex=True, na=False)
        criterio_txt = 'NO_MANTENEDORA'
    elif 'NO_IES' in df_teste.columns:
        no_ies_txt = df_teste['NO_IES'].astype('string').fillna('').str.upper()
        in_veduca = no_ies_txt.str.contains(r'V-EDUCA|VEDUCA', regex=True, na=False)
        criterio_txt = 'NO_IES'
    else:
        print('Destaque V-Educa nao aplicado: colunas NO_MANTENEDORA e NO_IES ausentes na base slim.')

    if criterio_txt:
        qtd_veduca = int(in_veduca.sum())
        if qtd_veduca == 0:
            print(f'Destaque V-Educa nao aplicado: nenhum registro identificado por {criterio_txt} na base slim.')
        else:
            modalidade_num = pd.to_numeric(df_teste['TP_MODALIDADE_ENSINO'], errors='coerce')
            df_teste.loc[in_veduca & (modalidade_num == 1), '__modal_label__'] = 'VEDUCA Presencial'
            df_teste.loc[in_veduca & (modalidade_num == 2), '__modal_label__'] = 'VEDUCA EAD'
            print(f'Destaque V-Educa ativo ({qtd_veduca:,} registros identificados por {criterio_txt} na base slim).')

    resumo_grupos = (
        df_teste.groupby('__modal_label__', observed=True)
        .agg(
            registros=('__modal_label__', 'size'),
            qt_mat=('QT_MAT', 'sum'),
            qt_ing=('QT_ING', 'sum'),
        )
        .sort_values('qt_mat', ascending=False)
    )

    print('')
    print('Resumo por grupo de modalidade com destaque V-Educa:')
    print(resumo_grupos)

    ordem = ['Presencial', 'Curso a distancia', 'VEDUCA Presencial', 'VEDUCA EAD', 'Outros']
    resumo_plot = resumo_grupos.reset_index().rename(columns={'__modal_label__': 'grupo'})
    resumo_plot['grupo'] = pd.Categorical(resumo_plot['grupo'], categories=ordem, ordered=True)
    resumo_plot = resumo_plot.sort_values('grupo')

    color_map = {
        'Presencial': '#2ca02c',
        'Curso a distancia': '#1f77b4',
        'VEDUCA Presencial': '#ff8c42',
        'VEDUCA EAD': '#ffb26b',
        'Outros': '#999999',
    }

    fig_teste = px.bar(
        resumo_plot,
        x='grupo',
        y='qt_mat',
        text='qt_mat',
        title='Teste de destaque V-Educa no INEP (Matriculados)',
        labels={'grupo': 'Grupo', 'qt_mat': 'Matriculados (QT_MAT)'},
        color='grupo',
        color_discrete_map=color_map,
    )
    fig_teste.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
    fig_teste.update_layout(showlegend=False, height=520)
    fig_teste.show()
else:
    print('Nao foi possivel validar o destaque V-Educa porque a coluna TP_MODALIDADE_ENSINO nao existe na base slim.')

Coluna de IES nao encontrada na base V-Educa extraida e lista manual vazia.
Colunas disponiveis:
['ANO', 'UF', 'AREA', 'CURSO', 'MODALIDADE', 'TICKET MÉDIO', 'INGRESSANTES', 'MATRICULADOS', 'NO_CINE_AREA_GERAL']


## 2. Exploração Inicial

Análise estrutural dos dados carregados

In [5]:
print("ESTRUTURA DA BASE")
print("=" * 80)
print(df_inep.info())
print("\n" + "=" * 80)
print(f"\nValores faltantes (%):")
missing_pct = (df_inep.isna().sum() / len(df_inep) * 100).sort_values(ascending=False)
print(missing_pct[missing_pct > 0].head(10))

ESTRUTURA DA BASE
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 722721 entries, 0 to 722720
Data columns (total 23 columns):
 #   Column                       Non-Null Count   Dtype 
---  ------                       --------------   ----- 
 0   NU_ANO_CENSO                 722721 non-null  Int64 
 1   SG_UF                        710943 non-null  string
 2   NO_REGIAO                    708571 non-null  string
 3   TP_REDE                      720349 non-null  Int64 
 4   NO_CINE_ROTULO               722721 non-null  string
 5   NO_CINE_AREA_GERAL           722721 non-null  string
 6   TP_MODALIDADE_ENSINO         722721 non-null  Int64 
 7   QT_CURSO                     720349 non-null  Int64 
 8   QT_VG_TOTAL                  720349 non-null  Int64 
 9   QT_VG_TOTAL_EAD              720349 non-null  Int64 
 10  QT_ING                       722721 non-null  Int64 
 11  QT_ING_FEM                   720349 non-null  Int64 
 12  QT_ING_MASC                  720349 non-null  Int64 
 

In [6]:
print("\nDISTRIBUIÇÃO POR MODALIDADE")
print("=" * 80)
print(df_inep['TP_MODALIDADE_ENSINO'].value_counts())

print("\n\nDISTRIBUIÇÃO POR REGIÃO")
print("=" * 80)
print(df_inep['NO_REGIAO'].value_counts())

print("\n\nDISTRIBUIÇÃO POR TIPO DE ORGANIZAÇÃO ACADÊMICA")
print("=" * 80)
print(df_inep['TP_ORGANIZACAO_ACADEMICA'].value_counts())

print("\n\nDISTRIBUIÇÃO POR CATEGORIA ADMINISTRATIVA")
print("=" * 80)
print(df_inep['TP_CATEGORIA_ADMINISTRATIVA'].value_counts())


DISTRIBUIÇÃO POR MODALIDADE
TP_MODALIDADE_ENSINO
2    687213
1     35508
Name: count, dtype: Int64


DISTRIBUIÇÃO POR REGIÃO
NO_REGIAO
Sudeste         291176
Sul             157232
Nordeste        142672
Centro-Oeste     60041
Norte            57450
Name: count, dtype: Int64


DISTRIBUIÇÃO POR TIPO DE ORGANIZAÇÃO ACADÊMICA
TP_ORGANIZACAO_ACADEMICA
2    370724
1    323102
3     23985
4      2474
5        64
Name: count, dtype: Int64


DISTRIBUIÇÃO POR CATEGORIA ADMINISTRATIVA
TP_CATEGORIA_ADMINISTRATIVA
4    638763
5     61435
1     10352
2      8608
7       936
3       255
Name: count, dtype: Int64


In [7]:
print("\nESTATÍSTICAS PRINCIPAIS")
print("=" * 80)
print(f"Total de Cursos: {df_inep['QT_CURSO'].sum():,}")
print(f"Total de Vagas (QT_VG_TOTAL): {df_inep['QT_VG_TOTAL'].sum():,}")
print(f"Total de Ingressantes (QT_ING): {df_inep['QT_ING'].sum():,}")
print(f"Total de Matriculados (QT_MAT): {df_inep['QT_MAT'].sum():,}")
print(f"Total de Concluintes (QT_CONC): {df_inep['QT_CONC'].sum():,}")

print("\n\nESTATÍSTICAS DESCRITIVAS")
print("=" * 80)
stats_cols = ['QT_CURSO', 'QT_VG_TOTAL', 'QT_ING', 'QT_MAT', 'QT_CONC']
print(df_inep[stats_cols].describe())


ESTATÍSTICAS PRINCIPAIS
Total de Cursos: 45,776
Total de Vagas (QT_VG_TOTAL): 23,665,419
Total de Ingressantes (QT_ING): 5,454,395
Total de Matriculados (QT_MAT): 11,158,764
Total de Concluintes (QT_CONC): 1,333,988


ESTATÍSTICAS DESCRITIVAS
       QT_CURSO  QT_VG_TOTAL    QT_ING      QT_MAT    QT_CONC
count  720349.0     720349.0  722721.0    722721.0   720349.0
mean   0.063547    32.852713  7.547027   15.439933   1.851863
std    0.243944    782.10785  50.78924  108.484483  11.021008
min         0.0          0.0       0.0         0.0        0.0
25%         0.0          0.0       0.0         1.0        0.0
50%         0.0          0.0       1.0         1.0        0.0
75%         0.0          0.0       3.0         5.0        1.0
max         1.0     100485.0   19169.0     29865.0     1549.0


## 3. Análises por Região

Distribuição de matrículas, ingressantes e vagas por região

In [8]:
# Agregação por região
regiao_agg = df_inep.groupby('NO_REGIAO', observed=True).agg({
    'QT_CURSO': 'sum',
    'QT_VG_TOTAL': 'sum',
    'QT_ING': 'sum',
    'QT_MAT': 'sum',
    'QT_CONC': 'sum'
}).sort_values('QT_MAT', ascending=False)

print("MATRÍCULAS POR REGIÃO")
print("=" * 80)
print(regiao_agg)

# Gráfico por modalidade e região
modalidade_regiao = df_inep.groupby(['NO_REGIAO', 'TP_MODALIDADE_ENSINO'], observed=True)['QT_MAT'].sum().reset_index()

fig = px.bar(
    modalidade_regiao,
    x='NO_REGIAO',
    y='QT_MAT',
    color='TP_MODALIDADE_ENSINO',
    title='Matrículas por Região e Modalidade',
    labels={'QT_MAT': 'Matrículas', 'NO_REGIAO': 'Região', 'TP_MODALIDADE_ENSINO': 'Modalidade'},
    barmode='stack'
)
fig.show()

MATRÍCULAS POR REGIÃO
              QT_CURSO  QT_VG_TOTAL   QT_ING   QT_MAT  QT_CONC
NO_REGIAO                                                     
Sudeste          14366      2309727  2337201  4518942   627003
Nordeste          7526      1227869   944841  2186521   263345
Sul               6389       621655   882203  1750161   231226
Centro-Oeste      3409       451417   460760   905093   115081
Norte             2789       464478   384111   863969    96860


## 4. Análise por Modalidade de Ensino

Presencial, EAD e Híbrida - Comparação de métricas

In [9]:
modalidade_agg = df_inep.groupby('TP_MODALIDADE_ENSINO', observed=True).agg({
    'QT_CURSO': 'sum',
    'QT_VG_TOTAL': 'sum',
    'QT_ING': 'sum',
    'QT_MAT': 'sum',
    'QT_CONC': 'sum'
}).sort_values('QT_MAT', ascending=False)

print("ESTATÍSTICAS POR MODALIDADE")
print("=" * 80)
print(modalidade_agg)
print("\n% de Matrículas:")
print((modalidade_agg['QT_MAT'] / modalidade_agg['QT_MAT'].sum() * 100).round(2))

# Gráfico pizza de distribuição por modalidade
fig_pizza = go.Figure(data=[go.Pie(
    labels=modalidade_agg.index,
    values=modalidade_agg['QT_MAT'],
    textposition='auto',
    textinfo='label+percent'
)])
fig_pizza.update_layout(title='Distribuição de Matrículas por Modalidade')
fig_pizza.show()

ESTATÍSTICAS POR MODALIDADE
                      QT_CURSO  QT_VG_TOTAL   QT_ING   QT_MAT  QT_CONC
TP_MODALIDADE_ENSINO                                                  
2                        11297     18590273  3624198  5612934   604742
1                        34479      5075146  1830197  5545830   729246

% de Matrículas:
TP_MODALIDADE_ENSINO
2    50.3
1    49.7
Name: QT_MAT, dtype: Float64


## 5. Análise por Categoria Administrativa

Pública x Privada (Lucrativa e Filantrópica)

In [10]:
categoria_agg = df_inep.groupby('TP_CATEGORIA_ADMINISTRATIVA', observed=True).agg({
    'QT_CURSO': 'sum',
    'QT_VG_TOTAL': 'sum',
    'QT_ING': 'sum',
    'QT_MAT': 'sum',
    'QT_CONC': 'sum'
}).sort_values('QT_MAT', ascending=False)

print("ESTATÍSTICAS POR CATEGORIA ADMINISTRATIVA")
print("=" * 80)
print(categoria_agg)
print("\n% de Matrículas:")
print((categoria_agg['QT_MAT'] / categoria_agg['QT_MAT'].sum() * 100).round(2))

ESTATÍSTICAS POR CATEGORIA ADMINISTRATIVA
                             QT_CURSO  QT_VG_TOTAL   QT_ING   QT_MAT  QT_CONC
TP_CATEGORIA_ADMINISTRATIVA                                                  
4                               22363     18315459  3714283  6372474   805313
5                               11932      4366942   721000  1789725   272168
1                                7124       586246   345453  1317548   157067
2                                3701       296641   201760   666980    88129
7                                 416        75669    16064    45400     6986
3                                 240        24462    12053    35139     4325

% de Matrículas:
TP_CATEGORIA_ADMINISTRATIVA
4    62.31
5     17.5
1    12.88
2     6.52
7     0.44
3     0.34
Name: QT_MAT, dtype: Float64


## 6. Resumo Executivo e Exportação

Consolidação dos principais insights

In [11]:
# Criar resumo executivo
resumo = f"""
RESUMO EXECUTIVO - MICRODADOS INEP 2024
{'='*80}

Data: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}

MÉTRICAS GLOBAIS
{'-'*80}
Total de Registros: {len(df_inep):,}
Total de Cursos: {df_inep['QT_CURSO'].sum():,}
Total de Vagas: {df_inep['QT_VG_TOTAL'].sum():,}
Total de Ingressantes: {df_inep['QT_ING'].sum():,}
Total de Matriculados: {df_inep['QT_MAT'].sum():,}
Total de Concluintes: {df_inep['QT_CONC'].sum():,}

DISTRIBUIÇÃO POR MODALIDADE
{'-'*80}
{modalidade_agg.to_string()}

DISTRIBUIÇÃO POR REGIÃO
{'-'*80}
{regiao_agg.to_string()}

DISTRIBUIÇÃO POR CATEGORIA ADMINISTRATIVA
{'-'*80}
{categoria_agg.to_string()}
"""

print(resumo)

# Salvar resumo em arquivo
resumo_path = OUTPUT_DIR / f"resumo_executivo_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(resumo_path, 'w', encoding='utf-8') as f:
    f.write(resumo)
print(f"\n✓ Resumo salvo em: {resumo_path}")

# Exportar agregações
regiao_agg.to_csv(OUTPUT_DIR / 'analise_por_regiao.csv', encoding='utf-8-sig')
modalidade_agg.to_csv(OUTPUT_DIR / 'analise_por_modalidade.csv', encoding='utf-8-sig')
categoria_agg.to_csv(OUTPUT_DIR / 'analise_por_categoria_administrativa.csv', encoding='utf-8-sig')
print("✓ Agregações exportadas em CSV")


RESUMO EXECUTIVO - MICRODADOS INEP 2024

Data: 09/04/2026 09:47:48

MÉTRICAS GLOBAIS
--------------------------------------------------------------------------------
Total de Registros: 722,721
Total de Cursos: 45,776
Total de Vagas: 23,665,419
Total de Ingressantes: 5,454,395
Total de Matriculados: 11,158,764
Total de Concluintes: 1,333,988

DISTRIBUIÇÃO POR MODALIDADE
--------------------------------------------------------------------------------
                      QT_CURSO  QT_VG_TOTAL   QT_ING   QT_MAT  QT_CONC
TP_MODALIDADE_ENSINO                                                  
2                        11297     18590273  3624198  5612934   604742
1                        34479      5075146  1830197  5545830   729246

DISTRIBUIÇÃO POR REGIÃO
--------------------------------------------------------------------------------
              QT_CURSO  QT_VG_TOTAL   QT_ING   QT_MAT  QT_CONC
NO_REGIAO                                                     
Sudeste          14366      2

## Próximos Passos

Possíveis análises futuras:

1. **Análises por Área CINE** - Distribuição de cursos e matrículas por área de conhecimento
2. **Análises por Tipo de Organização Acadêmica** - Universidades vs Faculdades
3. **Análises de Gratuidade** - Cursos gratuitos vs pagos
4. **Análises Demográficas** - Desagregação por sexo, raça/cor em ingressantes e matriculados
5. **Análises de Financiamento** - FIES, ProUni, bolsas
6. **Análises Longitudinais** - Comparação com anos anteriores
7. **Análises por UF** - Distribuição estadual detalhada

Arquivos gerados:
- `resumo_executivo_*.txt`
- `analise_por_regiao.csv`
- `analise_por_modalidade.csv`
- `analise_por_categoria_administrativa.csv`

## 7. Gráfico Dinâmico por Variável

Selecione uma variável numérica para análise estatística e uma dimensão categórica para comparação interativa.

In [12]:
# Grafico dinamico por variavel escolhida
from IPython.display import display
import ipywidgets as widgets

# Variaveis do dicionario para analise (Nome apropriado + codigo)
variaveis_dicionario = {
    'Ano do Censo (NU_ANO_CENSO)': 'NU_ANO_CENSO',
    'UF (SG_UF)': 'SG_UF',
    'Regiao (NO_REGIAO)': 'NO_REGIAO',
    'Nome da IES (NO_IES)': 'NO_IES',
    'Tipo de Rede (TP_REDE)': 'TP_REDE',  # opcional
    'Rotulo CINE (NO_CINE_ROTULO)': 'NO_CINE_ROTULO',
    'Area CINE Geral (NO_CINE_AREA_GERAL)': 'NO_CINE_AREA_GERAL',
    'Modalidade de Ensino (TP_MODALIDADE_ENSINO)': 'TP_MODALIDADE_ENSINO',
    'Quantidade de Cursos (QT_CURSO)': 'QT_CURSO',
    'Vagas Totais (QT_VG_TOTAL)': 'QT_VG_TOTAL',
    'Vagas Totais EAD (QT_VG_TOTAL_EAD)': 'QT_VG_TOTAL_EAD',
    'Ingressantes Total (QT_ING)': 'QT_ING',
    'Ingressantes Feminino (QT_ING_FEM)': 'QT_ING_FEM',
    'Ingressantes Masculino (QT_ING_MASC)': 'QT_ING_MASC',
    'Matriculados Total (QT_MAT)': 'QT_MAT',
    'Matriculados Feminino (QT_MAT_FEM)': 'QT_MAT_FEM',
    'Matriculados Masculino (QT_MAT_MASC)': 'QT_MAT_MASC',
}

# Mapeamento de codigos para exibicao amigavel nos filtros e graficos
value_label_maps = {
    'TP_REDE': {
        1: 'Publica',
        2: 'Privada',
    },
    'TP_MODALIDADE_ENSINO': {
        1: 'Presencial',
        2: 'Curso a distancia',
    },
}

def normalize_key(v):
    if pd.isna(v):
        return None
    try:
        return int(v)
    except Exception:
        return str(v)

def value_label(col, v):
    key = normalize_key(v)
    if col in value_label_maps and key in value_label_maps[col]:
        return f"{value_label_maps[col][key]} ({key})"
    return str(v)

variaveis_disponiveis = {k: v for k, v in variaveis_dicionario.items() if v in df_inep.columns}
col_to_label = {v: k for k, v in variaveis_disponiveis.items()}

if not variaveis_disponiveis:
    print('Nenhuma variavel do dicionario disponivel na base carregada.')
else:
    # Forca colunas codificadas como dimensoes para aparecerem nos filtros
    forced_dimension_cols = {'TP_REDE', 'TP_MODALIDADE_ENSINO'}

    # separa automaticamente variaveis numericas e categoricas com base na base atual
    vars_numericas = {}
    vars_categoricas = {}
    for label, col in variaveis_disponiveis.items():
        if col in forced_dimension_cols:
            vars_categoricas[label] = col
        if pd.api.types.is_numeric_dtype(df_inep[col]):
            vars_numericas[label] = col
        elif pd.api.types.is_string_dtype(df_inep[col]) or pd.api.types.is_categorical_dtype(df_inep[col]) or pd.api.types.is_object_dtype(df_inep[col]):
            vars_categoricas[label] = col

    # fallback de dimensoes para garantir pelo menos 1 opcao
    dims_base = vars_categoricas.copy()
    if 'NO_REGIAO' in variaveis_disponiveis.values() and 'Regiao (NO_REGIAO)' not in dims_base:
        dims_base['Regiao (NO_REGIAO)'] = 'NO_REGIAO'
    if 'SG_UF' in variaveis_disponiveis.values() and 'UF (SG_UF)' not in dims_base:
        dims_base['UF (SG_UF)'] = 'SG_UF'
    for forced_col in forced_dimension_cols:
        if forced_col in variaveis_disponiveis.values():
            forced_label = col_to_label.get(forced_col, forced_col)
            dims_base[forced_label] = forced_col

    available_numeric_vars = vars_numericas
    available_categorical_dims = dims_base

    if not available_numeric_vars:
        print('Sem variaveis numericas para plotar metricas.')
    elif not available_categorical_dims:
        print('Sem dimensoes categoricas para comparacao.')
    else:
        default_var = 'QT_MAT' if 'QT_MAT' in available_numeric_vars.values() else next(iter(available_numeric_vars.values()))
        default_dim = 'NO_REGIAO' if 'NO_REGIAO' in available_categorical_dims.values() else next(iter(available_categorical_dims.values()))

        w_variavel = widgets.Dropdown(
            options=list(available_numeric_vars.items()),
            value=default_var,
            description='Variavel',
            layout=widgets.Layout(width='420px')
        )

        w_dimensao = widgets.Dropdown(
            options=list(available_categorical_dims.items()),
            value=default_dim,
            description='Dimensao',
            layout=widgets.Layout(width='420px')
        )

        w_topn_var = widgets.IntSlider(
            value=10,
            min=5,
            max=30,
            step=1,
            description='Top N',
            continuous_update=False,
            layout=widgets.Layout(width='420px')
        )

        # Filtros adicionais (ate 3)
        no_filter = '__SEM_FILTRO__'
        extra_filter_options = [('Sem filtro', no_filter)] + list(available_categorical_dims.items())

        w_f1_dim = widgets.Dropdown(options=extra_filter_options, value=no_filter, description='Filtro 1', layout=widgets.Layout(width='420px'))
        w_f1_vals = widgets.SelectMultiple(options=[], value=(), description='Valores 1', layout=widgets.Layout(width='420px', height='120px'))

        w_f2_dim = widgets.Dropdown(options=extra_filter_options, value=no_filter, description='Filtro 2', layout=widgets.Layout(width='420px'))
        w_f2_vals = widgets.SelectMultiple(options=[], value=(), description='Valores 2', layout=widgets.Layout(width='420px', height='120px'))

        w_f3_dim = widgets.Dropdown(options=extra_filter_options, value=no_filter, description='Filtro 3', layout=widgets.Layout(width='420px'))
        w_f3_vals = widgets.SelectMultiple(options=[], value=(), description='Valores 3', layout=widgets.Layout(width='420px', height='120px'))

        out_var = widgets.Output()

        def format_series_stats(series: pd.Series) -> pd.DataFrame:
            s = pd.to_numeric(series, errors='coerce').dropna()
            if s.empty:
                return pd.DataFrame()
            stats = {
                'count': s.count(),
                'mean': s.mean(),
                'median': s.median(),
                'std': s.std(),
                'min': s.min(),
                'p25': s.quantile(0.25),
                'p75': s.quantile(0.75),
                'max': s.max(),
                'sum': s.sum(),
            }
            return pd.DataFrame(stats, index=[0]).T.rename(columns={0: 'valor'})

        def fmt_num(value):
            if pd.isna(value):
                return '-'
            return f'{value:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

        def refresh_filter_values(dim_widget, values_widget):
            selected_col = dim_widget.value
            if selected_col == no_filter:
                values_widget.options = []
                values_widget.value = ()
                return

            vals = df_inep[selected_col].dropna().unique().tolist()
            vals = sorted(vals, key=lambda x: str(x))
            options = [(value_label(selected_col, v), v) for v in vals]
            values_widget.options = options

            valid_values = set(vals)
            current = [v for v in values_widget.value if v in valid_values]
            values_widget.value = tuple(current)

        def apply_extra_filters(data):
            filtered = data
            filters = [
                (w_f1_dim.value, list(w_f1_vals.value)),
                (w_f2_dim.value, list(w_f2_vals.value)),
                (w_f3_dim.value, list(w_f3_vals.value)),
            ]
            for dim_col, selected_vals in filters:
                if dim_col != no_filter and selected_vals:
                    filtered = filtered[filtered[dim_col].isin(selected_vals)]
            return filtered

        def make_dim_label_series(series, dim_col):
            return series.map(lambda v: value_label(dim_col, v))

        def atualizar_grafico_variavel(*_):
            var_col = w_variavel.value
            dim_col = w_dimensao.value
            topn = w_topn_var.value
            var_label = col_to_label.get(var_col, var_col)
            dim_label = col_to_label.get(dim_col, dim_col)

            if var_col not in df_inep.columns or dim_col not in df_inep.columns:
                with out_var:
                    out_var.clear_output(wait=True)
                    print('Variavel ou dimensao nao existe na base carregada.')
                return

            df_work = apply_extra_filters(df_inep)
            if df_work.empty:
                with out_var:
                    out_var.clear_output(wait=True)
                    print('Sem dados apos aplicar os filtros adicionais.')
                return

            df_work = df_work[[var_col, dim_col]].copy()
            df_work[var_col] = pd.to_numeric(df_work[var_col], errors='coerce')
            df_work = df_work.dropna(subset=[var_col, dim_col])

            with out_var:
                out_var.clear_output(wait=True)
                if df_work.empty:
                    print('Sem dados validos para a variavel selecionada.')
                    return

                print(f'Variavel analisada: {var_label}')
                print(f'Dimensao de comparacao: {dim_label}')
                print(f'Registros validos: {len(df_work):,}'.replace(',', '.'))
                print('\nResumo estatistico:')

                stats_df = format_series_stats(df_work[var_col])
                if stats_df.empty:
                    print('Sem estatisticas disponiveis.')
                else:
                    display(stats_df.style.format(fmt_num))

                comp = (
                    df_work.groupby(dim_col, observed=True)[var_col]
                    .agg(['count', 'mean', 'median', 'sum'])
                    .reset_index()
                    .sort_values('sum', ascending=False)
                    .head(topn)
                )

                if comp.empty:
                    print('Sem categorias suficientes para gerar o grafico.')
                    return

                # Copias com labels amigaveis para os eixos
                comp_plot = comp.copy()
                comp_plot['dim_label'] = make_dim_label_series(comp_plot[dim_col], dim_col)
                order_labels = comp_plot['dim_label'].tolist()

                df_plot = df_work[df_work[dim_col].isin(comp[dim_col])].copy()
                df_plot['dim_label'] = make_dim_label_series(df_plot[dim_col], dim_col)

                fig_box = px.box(
                    df_plot,
                    x='dim_label',
                    y=var_col,
                    points='outliers',
                    title=f'Distribuicao de {var_label} por {dim_label}',
                    labels={var_col: var_label, 'dim_label': dim_label},
                    category_orders={'dim_label': order_labels}
                )
                fig_box.update_layout(height=500)
                fig_box.show()

                fig_bar_sum = px.bar(
                    comp_plot,
                    x='dim_label',
                    y='sum',
                    text='sum',
                    title=f'Total absoluto de {var_label} por {dim_label} (Top {topn})',
                    labels={'sum': f'Total de {var_label}', 'dim_label': dim_label},
                    category_orders={'dim_label': order_labels}
                )
                fig_bar_sum.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
                fig_bar_sum.update_layout(height=520)
                fig_bar_sum.show()

                fig_rank = px.bar(
                    comp_plot.sort_values('sum', ascending=True),
                    x='sum',
                    y='dim_label',
                    orientation='h',
                    text='sum',
                    title=f'Ranking de total absoluto de {var_label} por {dim_label}',
                    labels={'sum': f'Total de {var_label}', 'dim_label': dim_label}
                )
                fig_rank.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
                fig_rank.update_layout(height=520)
                fig_rank.show()

        def on_filter_dim_change(change, dim_widget, values_widget):
            if change.get('name') == 'value':
                refresh_filter_values(dim_widget, values_widget)
                atualizar_grafico_variavel()

        w_f1_dim.observe(lambda change: on_filter_dim_change(change, w_f1_dim, w_f1_vals), names='value')
        w_f2_dim.observe(lambda change: on_filter_dim_change(change, w_f2_dim, w_f2_vals), names='value')
        w_f3_dim.observe(lambda change: on_filter_dim_change(change, w_f3_dim, w_f3_vals), names='value')

        for widget_ in [w_variavel, w_dimensao, w_topn_var, w_f1_vals, w_f2_vals, w_f3_vals]:
            widget_.observe(atualizar_grafico_variavel, names='value')

        display(widgets.VBox([
            widgets.HTML('<h4>Grafico Dinamico por Variavel (Dicionario)</h4>'),
            widgets.HBox([w_variavel, w_dimensao]),
            w_topn_var,
            widgets.HTML('<b>Filtros adicionais (ate 3 dimensoes)</b>'),
            widgets.HBox([w_f1_dim, w_f1_vals]),
            widgets.HBox([w_f2_dim, w_f2_vals]),
            widgets.HBox([w_f3_dim, w_f3_vals]),
            out_var
        ]))

        refresh_filter_values(w_f1_dim, w_f1_vals)
        refresh_filter_values(w_f2_dim, w_f2_vals)
        refresh_filter_values(w_f3_dim, w_f3_vals)
        atualizar_grafico_variavel()